In [43]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import os
from tqdm import tqdm

from final_dataloader import is_place
from utils import *
from utils.config import device as DEVICE
from itertools import combinations

In [ ]:
data_x = np.load("final_loaded_data/location_ST_1400/weighed/train/data_x.npz")
data_y = np.load("final_loaded_data/location_ST_1400/weighed/train/data_y.npz")

In [ ]:
race_ids = list(data_x.keys())

In [ ]:
def to_tensor(arr, dtype=torch.float32, device=DEVICE):
    return torch.tensor(arr, dtype=dtype, device=device)

In [ ]:
def format_listwise_race(race_x):
    race_x = to_tensor(race_x, device="cpu")  # shape: (m, 71)
    m = race_x.size(0)
    assert m >= 4
    groups = list(combinations(range(m), 4))  # all index combinations of 4 horses

    # Stack each group of 4 horses into a flattened 284-dim vector (4 * 71)
    result = torch.stack([
        race_x[list(group)].reshape(-1) for group in groups  # shape: (284,)
    ])

    return result  # shape: (num_combinations, 284)

In [ ]:
race_ids = list(data_x.keys())
result = []
for race_id in tqdm(race_ids, desc="Loading data x"):
    race_x = format_listwise_race(data_x[race_id])
    result.append(race_x)

In [ ]:
data_x = torch.cat(result)

In [ ]:
def format_listwise_race_y(race_y):
    race_y = to_tensor(race_y, device="cpu")[:, :1]
    m = race_y.size(0)
    assert m >= 4
    groups = list(combinations(range(m), 4))
    result = torch.stack([
        race_y[list(group)].reshape(-1) for group in groups
    ])

    return torch.argmin(result, dim=1)

In [ ]:
result = []
for race_id in tqdm(race_ids, desc="Loading data y"):
    result.append(format_listwise_race_y(data_y[race_id]))
data_y = torch.cat(result)

In [ ]:
data_x = data_x.to(DEVICE)
data_y = data_y.to(DEVICE)
print(data_x.size())
print(data_y.size())


In [ ]:
class ListwiseModel(nn.Module):
    def __init__(self):
        super(ListwiseModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 4),
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
def shuffle_indices(m, cv_ratio=0.2):
    indices = torch.randperm(m)
    cv_idx = int(m * (1 - cv_ratio))
    return indices[:cv_idx], indices[cv_idx:]

In [ ]:
train_idx, cv_idx = shuffle_indices(data_x.size(0))
train_x = data_x[train_idx]
train_y = data_y[train_idx]
cv_x = data_x[cv_idx]
cv_y = data_y[cv_idx]

In [ ]:
train_mean = torch.mean(train_x, dim=0)
train_std = torch.std(train_x, dim=0)
train_std[train_std == 0] = 1

train_x = (train_x - train_mean) / train_std
cv_x = (cv_x - train_mean) / train_std

In [ ]:
model = ListwiseModel().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

In [ ]:
epochs = 5000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(train_x)
    loss = criterion(pred, train_y)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        cv_pred = model(cv_x)
        cv_loss = criterion(cv_pred, cv_y)
        print(f"{epoch + 1}/{epochs}: train loss = {loss.item()}, cv loss = {cv_loss.item()}")

In [ ]:
test_x = np.load("final_loaded_data/location_ST_1400/weighed/train/data_x.npz")
test_y = np.load("final_loaded_data/location_ST_1400/weighed/train/data_y.npz")

In [ ]:
race_ids = list(test_x.keys())
wins = 0
places = 0
total = 0
model.eval()
for race_id in tqdm(race_ids, desc="Simulating races"):
    race_x = test_x[race_id]
    m = race_x.shape[0]
    listwise_x = format_listwise_race(race_x)
    listwise_x = listwise_x.to(DEVICE)

    output = model(listwise_x)
    scores = torch.softmax(output, dim=1)

    scores_sum = torch.zeros(m, device=DEVICE)
    scores_count = torch.zeros(m, device=DEVICE)

    combos = list(combinations(range(m), 4))

    for i, idxs in enumerate(combos):
        for j, horse_idx in enumerate(idxs):
            scores_sum[horse_idx] += scores[i, j]
            scores_count[horse_idx] += 1

    average_scores = scores_sum / scores_count
    predicted_winner = torch.argmax(average_scores)

    race_y = test_y[race_id]
    race_y = to_tensor(race_y)

    if race_y[predicted_winner, 0] == 1:
        wins += 1
    if race_y[predicted_winner, 0] <= 3:
        places += 1

    total += 1



In [40]:
win_acc = wins / total
places_acc = places / total

print(f"WIN accuracy: {win_acc}")
print(f"PLACE accuracy: {places_acc}")

WIN accuracy: 0.0761904761904762
PLACE accuracy: 0.2571428571428571


tensor([[60.0000,  1.0000,  8.0000,  ...,  0.4652,  0.3961,  0.3955],
        [60.0000,  1.0000,  8.0000,  ...,  0.1800,  0.2975,  0.3193],
        [60.0000,  1.0000,  8.0000,  ...,  0.2846,  0.3058,  0.2986],
        ...,
        [81.0000, 10.0000,  2.0000,  ...,  0.0840,  0.1240,  0.1209],
        [81.0000, 10.0000,  2.0000,  ...,  0.0840,  0.1240,  0.1209],
        [81.0000, 11.0000, 11.0000,  ...,  0.0840,  0.1240,  0.1209]],
       device='cuda:0')